# PFE ML — Period-Fix Rebuild

This notebook rebuilds the clean layer and the company-year features so the four INSEE identity columns (`activity_code`, `legal_category_code`, `employee_size_bracket`, `administrative_status_at_cutoff`) become temporally valid period-dated features, then retrains the continuity-risk model.

Run this once after pulling the `data-extraction` branch with the period-fix commits. After it completes, `training_only.ipynb` can be used again for cheap retraining loops on the rebuilt feature table.

## 1. Runtime

Use **High-RAM CPU**. GPU/TPU is unused. The clean and feature builds are DuckDB-heavy and benefit from RAM.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
SMOKE_MAX_COMPANIES = 50_000
TARGET = 'continuity_risk_12m_label'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('BRANCH       =', BRANCH)
print('YEARS        =', START_YEAR, '-', END_YEAR)

## 2. Pull Code And Install Dependencies

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

## 3. Verify The INSEE Historique Raw Export Is Present

The fix depends on `data-lake/raw/insee/bulk/stock_unite_legale_historique/`. That file is in `INSEE_RESOURCES` and was downloaded the first time you ran the full pipeline. If this cell reports it missing, run `collabs/export_raw_sources.py --insee` once before continuing.

In [ ]:
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
HIST_ROOT = Path(f'{DATA_LAKE}/raw/insee/bulk/stock_unite_legale_historique')

found = list(HIST_ROOT.rglob('*.parquet')) if HIST_ROOT.exists() else []
if not found:
    raise SystemExit(
        f'Missing INSEE historique parquet under {HIST_ROOT}. '
        'Run once: python -m collabs.export_raw_sources --insee'
    )
print(f'Historique raw exports found ({len(found)} parquet file(s)):')
for p in found[:5]:
    print(' ', p)

## 4. Rebuild The Clean Layer

Calls `build_clean_core_sources` with `--overwrite`, which rebuilds all four clean datasets atomically (`company_identity`, `legal_events`, `formalities_events`, `annual_accounts`). Only `company_identity` is changed by this fix; the others are regenerated unchanged.

After it completes the cell asserts that `company_identity`'s manifest reports `schema_version: 2` and a period grain.

In [ ]:
import json, shlex, subprocess, sys

clean_cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.build_clean_core_sources',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--overwrite',
]
print(' '.join(shlex.quote(p) for p in clean_cmd))
subprocess.run(clean_cmd, check=True)

manifest_path = Path(f'{DATA_LAKE}/clean/company_identity/_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(manifest, indent=2))

assert manifest.get('schema_version') == 2, (
    f"company_identity schema_version expected 2, got {manifest.get('schema_version')!r}"
)
assert 'period' in manifest.get('grain', ''), (
    f"company_identity grain expected period-aware, got {manifest.get('grain')!r}"
)
print('\nClean rebuild verified: company_identity is now period-grained.')

## 5. Smoke-Test The Feature Build

A small `--max-companies` run catches SQL or column-name issues cheaply before the full multi-year rebuild. The temporal-validity check at the end counts SIRENs whose `activity_code` differs across prediction years. With the old SIREN-snapshot grain this count was always 0 (same value broadcast everywhere); with the period fix in effect it must be positive.

In [ ]:
smoke_feat_cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.build_company_year_features',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--start-year', str(START_YEAR),
    '--end-year', str(END_YEAR),
    '--max-companies', str(SMOKE_MAX_COMPANIES),
    '--overwrite',
]
print(' '.join(shlex.quote(p) for p in smoke_feat_cmd))
subprocess.run(smoke_feat_cmd, check=True)

import duckdb
FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

con = duckdb.connect()

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()
print('Feature table columns:')
print(schema[['column_name', 'column_type']].to_string(index=False))

expected = {
    'siren', 'prediction_year', 'prediction_date', 'activity_code',
    'legal_category_code', 'employee_size_bracket',
    'administrative_status_at_cutoff', 'company_age_years',
    'legal_events_count_all', 'latest_revenue',
}
missing = expected - set(schema['column_name'])
assert not missing, f'Missing expected feature columns: {missing}'
print('\nAll expected columns present.')

validity = con.execute(f'''
    WITH per_siren AS (
        SELECT siren, count(DISTINCT activity_code) AS distinct_codes
        FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)
        WHERE activity_code IS NOT NULL
        GROUP BY siren
    )
    SELECT
        count(*) AS total_sirens_with_activity,
        sum(CASE WHEN distinct_codes > 1 THEN 1 ELSE 0 END) AS sirens_with_changed_activity
    FROM per_siren
''').df()
print('\nTemporal validity check (activity_code variation across years):')
print(validity.to_string(index=False))
changed = int(validity['sirens_with_changed_activity'].iloc[0])
print('Interpretation:')
if changed == 0:
    print('  WARNING: 0 SIRENs changed activity_code across years.')
    print('  Either the clean layer is still SIREN-snapshot grain, or this 50k sample is too small.')
else:
    print(f'  OK: {changed:,} SIRENs show activity_code variation across prediction years.')
    print('  Period-based feature selection is in effect.')

con.close()

## 6. Full Feature Build

Rebuilds `company_year_features`, `risk_labels`, `company_features` for the full company set across 2017–2024. Expect roughly 15–30 minutes on Colab High-RAM CPU. If memory pressure is a concern, add `--year-batch-size 2` to the command below.

In [ ]:
full_feat_cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.build_company_year_features',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--start-year', str(START_YEAR),
    '--end-year', str(END_YEAR),
    '--overwrite',
]
print(' '.join(shlex.quote(p) for p in full_feat_cmd))
subprocess.run(full_feat_cmd, check=True)

con = duckdb.connect()
counts = con.execute(f'''
    SELECT
        (SELECT count(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS feature_rows,
        (SELECT count(*) FROM read_parquet('{LABELS_GLOB}',   union_by_name=true)) AS label_rows,
        (SELECT min(prediction_year) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS min_year,
        (SELECT max(prediction_year) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)) AS max_year
''').df()
print(counts.to_string(index=False))
con.close()

## 7. Train Run 5 (Period-Valid INSEE Features)

2M rows, same cap as Run 3, so the comparison isolates the effect of restoring the INSEE features as period-dated.

In [ ]:
train_cmd = [
    sys.executable, '-u',
    '-m', 'app.tools.train_continuity_model',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
    '--target', TARGET,
    '--train-start-year', str(START_YEAR),
    '--train-end-year', str(END_YEAR),
    '--max-rows', str(TRAIN_MAX_ROWS),
    '--min-rows', '1000',
]
print(' '.join(shlex.quote(p) for p in train_cmd))
subprocess.run(train_cmd, check=True)

metadata_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
print(json.dumps({
    'run_name': metadata.get('run_name'),
    'model_version': metadata.get('model_version'),
    'rows': metadata.get('rows'),
    'feature_count': metadata.get('feature_count'),
    'split_strategy': metadata.get('split_strategy'),
    'test_class_counts': metadata.get('test_class_counts'),
    'metrics': {
        k: metadata.get('metrics', {}).get(k)
        for k in ('accuracy', 'roc_auc', 'average_precision',
                  'precision_at_0_5', 'recall_at_0_5', 'f1_at_0_5')
    },
    'run_artifacts_dir': metadata.get('run_artifacts_dir'),
}, indent=2))

## 8. Display Run Artifacts

In [ ]:
from IPython.display import Image, Markdown, display

run_dir = Path(metadata['run_artifacts_dir'])
print('Run folder:')
print(run_dir)

print('\nFiles:')
for path in sorted(run_dir.iterdir()):
    print(path.name)

summary_path = run_dir / 'run_summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))

image_names = [
    'class_counts_by_year.png',
    'precision_recall_curve.png',
    'roc_curve.png',
    'confusion_matrix_at_0_5.png',
    'score_distribution_by_class.png',
    'threshold_tradeoff.png',
    'top_feature_coefficients.png',
]
for image_name in image_names:
    image_path = run_dir / image_name
    if image_path.exists():
        print('\n' + image_name)
        display(Image(filename=str(image_path)))

comparison_image = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.png'
if comparison_image.exists():
    print('\nmodel_run_comparison.png')
    display(Image(filename=str(comparison_image)))

## 9. What To Send After The Run

- `clean/company_identity/_manifest.json` (must show `schema_version: 2` and a period grain)
- The newest folder under `ml-artifacts/runs/`: `run_summary.md`, `precision_recall_curve.png`, `threshold_tradeoff.png`, `feature_coefficients.csv`
- `ml-artifacts/model_run_comparison.png` (it now shows the period-fix run alongside the earlier runs)
- Note in the report whether the temporal-validity smoke check in section 5 reported a positive `sirens_with_changed_activity`